In [1]:
import math
from collections import deque

GRID_SIZE = 5
START     = (1, 1)
TARGET    = (4, 4)
OBSTACLES = {(2, 2), (3, 1)}
DIAGONALS = [(-1, -1), (-1, +1), (+1, -1), (+1, +1)]
DIAGONAL_COST = math.sqrt(2)

def is_valid(r, c):
    return 0 <= r < GRID_SIZE and 0 <= c < GRID_SIZE and (r, c) not in OBSTACLES

def find_shortest_path():
    queue   = deque()
    visited = set()
    queue.append((START, [START]))
    visited.add(START)
    while queue:
        (row, col), path = queue.popleft()
        if (row, col) == TARGET:
            return path
        for dr, dc in DIAGONALS:
            nr, nc = row + dr, col + dc
            if is_valid(nr, nc) and (nr, nc) not in visited:
                visited.add((nr, nc))
                queue.append(((nr, nc), path + [(nr, nc)]))
    return None

def verify_path(path):
    if path[0] != START:
        return False, "Start cell mismatch"
    if path[-1] != TARGET:
        return False, "Target cell mismatch"
    for cell in path:
        if cell in OBSTACLES:
            return False, f"Path passes through obstacle {cell}"
    for i in range(len(path) - 1):
        r1, c1 = path[i]
        r2, c2 = path[i + 1]
        if (abs(r2 - r1), abs(c2 - c1)) != (1, 1):
            return False, f"Non-diagonal move between {path[i]} and {path[i+1]}"
    if len(path) != len(set(path)):
        return False, "Path contains a loop"
    return True, "All constraints satisfied"

def display_grid(path):
    path_set = set(path)
    print("\n  Grid (5x5):")
    print("  " + "  ".join(str(c) for c in range(GRID_SIZE)))
    for r in range(GRID_SIZE):
        row_str = f"{r} "
        for c in range(GRID_SIZE):
            cell = (r, c)
            if cell == START:
                row_str += " S "
            elif cell == TARGET:
                row_str += " T "
            elif cell in OBSTACLES:
                row_str += " X "
            elif cell in path_set:
                row_str += " * "
            else:
                row_str += " . "
        print(row_str)
    print("\n  Legend: S=Start  T=Target  X=Obstacle  *=Path  .=Empty")

def main():
    print("=" * 55)
    print("  Task 01: Warehouse Robot Navigation CSP")
    print("=" * 55)
    print(f"\nGrid size : {GRID_SIZE}x{GRID_SIZE}")
    print(f"Start     : {START}")
    print(f"Target    : {TARGET}")
    print(f"Obstacles : {sorted(OBSTACLES)}")

    print("\nVariables  : Position of robot at each time step (row, col)")
    print(f"Domains    : All valid grid cells ({GRID_SIZE*GRID_SIZE - len(OBSTACLES)} cells)")
    print("Constraints:")
    print("  (1) path[0] = (1,1) and path[-1] = (4,4)")
    print("  (2) Each consecutive move is diagonal (delta_row=+-1, delta_col=+-1)")
    print("  (3) No cell in path is an obstacle")
    print("  (4) No cell visited more than once")

    path = find_shortest_path()

    if path is None:
        print("No valid path found!")
        return

    ok, msg = verify_path(path)
    print(f"\nConstraint check : {msg}")
    print(f"Shortest path    : {' -> '.join(str(p) for p in path)}")
    print(f"Steps taken      : {len(path) - 1}")
    print(f"Total cost       : {len(path)-1} x sqrt(2) = {(len(path)-1)*DIAGONAL_COST:.4f}")
    display_grid(path)

main()

  Task 01: Warehouse Robot Navigation CSP

Grid size : 5x5
Start     : (1, 1)
Target    : (4, 4)
Obstacles : [(2, 2), (3, 1)]

Variables  : Position of robot at each time step (row, col)
Domains    : All valid grid cells (23 cells)
Constraints:
  (1) path[0] = (1,1) and path[-1] = (4,4)
  (2) Each consecutive move is diagonal (delta_row=+-1, delta_col=+-1)
  (3) No cell in path is an obstacle
  (4) No cell visited more than once

Constraint check : All constraints satisfied
Shortest path    : (1, 1) -> (0, 2) -> (1, 3) -> (2, 4) -> (3, 3) -> (4, 4)
Steps taken      : 5
Total cost       : 5 x sqrt(2) = 7.0711

  Grid (5x5):
  0  1  2  3  4
0  .  .  *  .  . 
1  .  S  .  *  . 
2  .  .  X  .  * 
3  .  X  .  *  . 
4  .  .  .  .  T 

  Legend: S=Start  T=Target  X=Obstacle  *=Path  .=Empty


In [2]:
from ortools.sat.python import cp_model
from collections import deque

GRID = [
    [0, 1, 1, 0, 0],
    [1, 1, 0, 0, 1],
    [0, 1, 1, 1, 0],
    [0, 0, 1, 0, 0],
    [1, 0, 0, 1, 1],
]

ROWS = len(GRID)
COLS = len(GRID[0])
DIRECTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def find_all_regions():
    visited = [[False] * COLS for _ in range(ROWS)]
    regions = []
    for start_r in range(ROWS):
        for start_c in range(COLS):
            if GRID[start_r][start_c] == 1 and not visited[start_r][start_c]:
                region = []
                queue  = deque([(start_r, start_c)])
                visited[start_r][start_c] = True
                while queue:
                    r, c = queue.popleft()
                    region.append((r, c))
                    for dr, dc in DIRECTIONS:
                        nr, nc = r + dr, c + dc
                        if (0 <= nr < ROWS and 0 <= nc < COLS
                                and not visited[nr][nc]
                                and GRID[nr][nc] == 1):
                            visited[nr][nc] = True
                            queue.append((nr, nc))
                regions.append(region)
    return regions

def compute_perimeter(region):
    land_set  = set(region)
    perimeter = 0
    for (r, c) in region:
        for dr, dc in DIRECTIONS:
            if (r + dr, c + dc) not in land_set:
                perimeter += 1
    return perimeter

def solve_with_ortools(largest_region):
    model  = cp_model.CpModel()
    solver = cp_model.CpSolver()

    x = {}
    for r in range(ROWS):
        for c in range(COLS):
            x[(r, c)] = model.new_int_var(0, 1, f"cell_{r}_{c}")
            model.add(x[(r, c)] == GRID[r][c])

    edge_vars = []
    land_set  = set(largest_region)

    for (r, c) in largest_region:
        for dr, dc in DIRECTIONS:
            nr, nc = r + dr, c + dc
            edge = model.new_bool_var(f"edge_{r}_{c}_{dr}_{dc}")
            if 0 <= nr < ROWS and 0 <= nc < COLS:
                model.add(x[(r, c)] - x[(nr, nc)] == 1).only_enforce_if(edge)
                model.add(x[(r, c)] - x[(nr, nc)] != 1).only_enforce_if(edge.Not())
            else:
                model.add(x[(r, c)] == 1).only_enforce_if(edge)
                model.add(x[(r, c)] != 1).only_enforce_if(edge.Not())
            edge_vars.append(edge)

    status = solver.solve(model)
    if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        return sum(solver.value(e) for e in edge_vars)
    return None

def display_grid(largest_region):
    land_set = set(largest_region)
    print("\n  Grid Map  (L=largest landmass, 1=other land, 0=water)")
    print("  " + "  ".join(str(c) for c in range(COLS)))
    for r in range(ROWS):
        row_str = f"{r} "
        for c in range(COLS):
            if (r, c) in land_set:
                row_str += " L "
            elif GRID[r][c] == 1:
                row_str += " 1 "
            else:
                row_str += " 0 "
        print(row_str)

def main():
    print("=" * 55)
    print("  Task 02: Satellite Island Perimeter CSP")
    print("=" * 55)

    print(f"\nVariables  : Binary cell x[r][c] for each of {ROWS}x{COLS} cells")
    print("Domains    : {0, 1}  (0=water, 1=land)")
    print("Constraints:")
    print("  (1) x[r][c] must equal the observed grid value")
    print("  (2) A boundary edge exists when land cell touches water or border")
    print("  (3) Connectivity enforced via BFS")

    regions = find_all_regions()
    print(f"\nLand regions found : {len(regions)}")
    for i, region in enumerate(regions):
        print(f"  Region {i+1}: {len(region)} cells -> {sorted(region)}")

    largest = max(regions, key=len)
    print(f"\nLargest landmass : {len(largest)} cells -> {sorted(largest)}")
    display_grid(largest)

    perimeter = solve_with_ortools(largest)
    if perimeter is not None:
        print(f"\nPerimeter (OR-Tools) : {perimeter} boundary edges")

    direct = compute_perimeter(largest)
    print(f"Perimeter (direct)   : {direct} boundary edges")

main()

ModuleNotFoundError: No module named 'ortools'

In [3]:
import math
import random
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

random.seed(42)
NUM_CITIES = 10
CITY_NAMES = [f"City_{i}" for i in range(NUM_CITIES)]
CITIES = [(random.randint(0, 100), random.randint(0, 100)) for _ in range(NUM_CITIES)]

def euclidean_distance(p1, p2):
    return math.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

def build_distance_matrix(cities):
    n = len(cities)
    dist = [[0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            dist[i][j] = int(euclidean_distance(cities[i], cities[j]))
    return dist

DISTANCE_MATRIX = build_distance_matrix(CITIES)

def solve_tsp():
    manager = pywrapcp.RoutingIndexManager(NUM_CITIES, 1, 0)
    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node   = manager.IndexToNode(to_index)
        return DISTANCE_MATRIX[from_node][to_node]

    transit_callback_index = routing.RegisterTransitCallback(distance_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    search_params = pywrapcp.DefaultRoutingSearchParameters()
    search_params.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    )
    search_params.local_search_metaheuristic = (
        routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    )
    search_params.time_limit.FromSeconds(5)

    solution = routing.SolveWithParameters(search_params)
    return manager, routing, solution

def extract_route(manager, routing, solution):
    index = routing.Start(0)
    route = []
    while not routing.IsEnd(index):
        route.append(manager.IndexToNode(index))
        index = solution.Value(routing.NextVar(index))
    route.append(manager.IndexToNode(index))
    return route

def print_route(route):
    total = 0
    for i in range(len(route) - 1):
        a, b = route[i], route[i + 1]
        d    = DISTANCE_MATRIX[a][b]
        total += d
        arrow = "->" if i < len(route) - 2 else "back to start"
        print(f"    {CITY_NAMES[a]:8s} {arrow} {CITY_NAMES[b]:8s}  (dist = {d})")
    print(f"\n  Total distance : {total} units")
    return total

def main():
    print("=" * 55)
    print("  Task 03: Travelling Salesman Problem CSP")
    print("=" * 55)

    print(f"\nVariables  : next[i] = city visited after city i,  i in 0..{NUM_CITIES-1}")
    print(f"Domains    : {{0, 1, ..., {NUM_CITIES-1}}}")
    print("Constraints:")
    print("  (1) next[i] != i  (no self-loop)")
    print("  (2) Each city visited exactly once  (all-different)")
    print("  (3) Route forms a single Hamiltonian cycle")
    print("Objective  : Minimize total travel distance")

    print("\nCity Coordinates:")
    for i, (x, y) in enumerate(CITIES):
        print(f"  {CITY_NAMES[i]:8s} : ({x:3d}, {y:3d})")

    manager, routing, solution = solve_tsp()

    if solution:
        route = extract_route(manager, routing, solution)
        print("\nOptimal Tour:")
        print_route(route)
        print(f"\n  Path : {' -> '.join(str(c) for c in route)}")
    else:
        print("No solution found.")

main()

ModuleNotFoundError: No module named 'ortools'